In [1]:
11

11

In [13]:
%load_ext autoreload
%autoreload 2

import sentiments_utils as utils
import torch
import os
from datasets import load_dataset, DatasetDict, Dataset, concatenate_datasets, disable_progress_bar
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    set_seed,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score, confusion_matrix
import numpy as np
from huggingface_hub import notebook_login
notebook_login()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
print(f"📊 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

📊 Device: GPU
   GPU: NVIDIA GeForce RTX 4070 Ti SUPER
   Memory: 17.17 GB


In [4]:
# Dataset configuration
DATASET_PATH = "sentiment_datasets"
DATASET_SPLITS = {"train": f"{DATASET_PATH}/train.jsonl",
                  "validation": f"{DATASET_PATH}/val.jsonl",
                  "test": f"{DATASET_PATH}/test.jsonl"}
# Language configuration
SOURCE_COLUMN = "shp" 
TARGET_COLUMN = "spa"
NUMBER_LABELS = 3  # Positive, Negative, Neutral
# General configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training configuration
CONFIDENCE_THRESHOLD = 0.6
MAX_LENGTH = 128
SEED = 120
set_seed(SEED)

# 1. Preparar Datos

In [5]:
# Cargar datasets
full_dataset = load_dataset("json", data_files=DATASET_SPLITS)
train_dataset = full_dataset["train"]
validation_dataset = full_dataset["validation"]
test_dataset = full_dataset["test"]

# 2. Preparar modelos

### Modelo 1: mmBERT-base


In [6]:
# Modelo base
base_model_name_1 =  "jhu-clsp/mmBERT-base"
base_tokenizer_name_1 =  "jhu-clsp/mmBERT-base"
base_model_path_1 = f"models/{base_model_name_1}"
base_tokenizer_path_1 = f"tokenizers/{base_model_name_1}"

# Modelo generado
trained_model_name_1 =  "mmBERT-sentiment-shp"
trained_model_path_1 = f"models/{trained_model_name_1}"
trained_tokenizer_path_1 = f"tokenizers/{trained_model_name_1}"

In [7]:
# Download Base model and tokenizer if not already present
if not os.path.exists(base_model_path_1):
    utils.download_model(base_model_name_1, base_model_path_1)
base_model_1 = utils.prepare_model(base_model_path_1, NUMBER_LABELS)
if not os.path.exists(base_tokenizer_path_1):
    utils.download_tokenizer(base_tokenizer_name_1, base_tokenizer_path_1)
base_tokenizer_1 = utils.prepare_tokenizer(base_tokenizer_path_1)
print("Model num_labels:", base_model_1.config.num_labels)


⬇️  Loading model: models/jhu-clsp/mmBERT-base


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at models/jhu-clsp/mmBERT-base and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Model loaded from: models/jhu-clsp/mmBERT-base

🔧 Loading tokenizer: tokenizers/jhu-clsp/mmBERT-base
   Tokenizer loaded from: tokenizers/jhu-clsp/mmBERT-base
Model num_labels: 3


### Modelo 2: xlm-roBERTa-base

In [8]:
# Modelo base
base_model_name_2 =  "FacebookAI/xlm-roberta-base"
base_tokenizer_name_2 =  "FacebookAI/xlm-roberta-base"
base_model_path_2 = f"models/{base_model_name_2}"
base_tokenizer_path_2 = f"tokenizers/{base_model_name_2}"

# Modelo generado
trained_model_name_2 =  "xlm-roberta-sentiment-shp-tokenizer-vocab"

In [9]:
# Download Base model and tokenizer if not already present
if not os.path.exists(base_model_path_2):
    utils.download_model(base_model_name_2, base_model_path_2)
base_model_2 = utils.prepare_model(base_model_path_2, NUMBER_LABELS)
if not os.path.exists(base_tokenizer_path_2):
    utils.download_tokenizer(base_tokenizer_name_2, base_tokenizer_path_2)
base_tokenizer_2 = utils.prepare_tokenizer(base_tokenizer_path_2)
print("Model num_labels:", base_model_2.config.num_labels)


⬇️  Loading model: models/FacebookAI/xlm-roberta-base


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at models/FacebookAI/xlm-roberta-base and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Model loaded from: models/FacebookAI/xlm-roberta-base

🔧 Loading tokenizer: tokenizers/FacebookAI/xlm-roberta-base
   Tokenizer loaded from: tokenizers/FacebookAI/xlm-roberta-base
Model num_labels: 3


# 3. Entrenar modelos

In [10]:
# Training hyperparameters
TRAIN_BATCH_SIZE = 4           # Adjust based on GPU memory
EVAL_BATCH_SIZE = 4          # Adjust based on GPU memory
LEARNING_RATE = 1e-5
NUM_EPOCHS = 40
WARMUP_STEPS = 500

In [11]:
# Training arguments
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=6, 
    early_stopping_threshold=0.001,
)

training_args = TrainingArguments(
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=WARMUP_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",
    greater_is_better=True,
    warmup_ratio=0.1,
    logging_steps=50,
    report_to="none",  # Disable wandb/tensorboard
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available,
    remove_unused_columns=False,  # Important for custom collator
)

callbacks=[early_stopping_callback]

In [12]:
def train_model(base_model, base_tokenizer, train_dataset, val_dataset, test_dataset, model_name, training_args : TrainingArguments, callbacks, output_dir = "models",  vocab_file=None):
    """Train the sentiment analysis model."""
    
    
    utils.clear_gpu_memory()

    
    def preprocess_function(dataset: Dataset):
        """Tokenize and encode the Shipibo texts."""
        encodings = base_tokenizer(dataset["shp"], truncation=True, padding=False, max_length=256)
        # Map sentiment labels to IDs
        label_map = {'NEG': 0, 'NEU': 1, 'POS': 2}
        encodings['label'] = [label_map[label] for label in dataset['sentiment']]
        return encodings

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        accuracy = accuracy_score(labels, predictions)
        precision = precision_score(labels, predictions, average='weighted', zero_division=0)
        recall = recall_score(labels, predictions, average='weighted', zero_division=0)
        f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
        balanced_acc = balanced_accuracy_score(labels, predictions)
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'balanced_accuracy': balanced_acc
        }
    
    def prepare_dataset(dataset: Dataset):
        """Prepare dataset for training/evaluation."""
        dataset = dataset.map(preprocess_function, batched=True)
        columns_to_remove = [ x for x in dataset.column_names if x not in ['input_ids', 'attention_mask', 'label']]
        dataset = dataset.remove_columns(columns_to_remove)
        print("Dataset labels:", set(dataset["label"]))
        return dataset
    
    def load_vocab_tokens(vocab_file):
        tokens = []
        with open(vocab_file) as f:
            for line in f:
                tok = line.split("\t")[0]
                tokens.append(tok)
        return tokens

    
    def extend_tokenizer_vocab(tokenizer, vocab_file):
        """Extend tokenizer vocabulary with additional tokens from vocab_file."""
        if vocab_file:
            print(f"🔧 Reescalando tokenizador y modelo con vocabulario de: {vocab_file}...", flush=True)
            vocab_tokens = load_vocab_tokens(vocab_file)
            vocab_tokens = [t for t in vocab_tokens if t not in tokenizer.get_vocab()]        
            num_added_toks = tokenizer.add_tokens(vocab_tokens)
            base_model.resize_token_embeddings(len(tokenizer))
            print(f"✅ Añadidos {num_added_toks} tokens especiales\n", flush=True)

                     
            with open(vocab_file, 'r', encoding='utf-8') as vf:
                new_tokens = [line.strip() for line in vf if line.strip() not in tokenizer.get_vocab()]
            if new_tokens:
                tokenizer.add_tokens(new_tokens)
                base_model.resize_token_embeddings(len(tokenizer))
                print(f"Extended tokenizer vocabulary by {len(new_tokens)} tokens.")

    
    # Tokenize datasets
    tokenized_train = prepare_dataset(train_dataset)
    tokenized_validation = prepare_dataset(val_dataset)
    tokenized_test = prepare_dataset(test_dataset)
    
    # Output path
    output_path = os.path.join(output_dir, model_name)
      
    # Initialize Trainer
    training_args.output_dir = output_path
    trainer = Trainer(
        model=base_model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_validation,
        tokenizer=base_tokenizer,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    # Train the model
    train_result = trainer.train()

    # Información sobre early stopping
    if hasattr(trainer.state, 'best_metric'):
        print(f"\n🏆 Mejor eval_loss: {trainer.state.best_metric:.6f}", flush=True)
        print(f"📍 Alcanzado en época: {train_result.metrics.get('epoch', 'N/A')}", flush=True)

    # Evaluate on test set
    test_results = trainer.evaluate(eval_dataset=tokenized_test)
    print("Test Results:", test_results)

    # Save the trained model and tokenizer
    trainer.save_model(output_path)
    base_tokenizer.save_pretrained(output_path)
    model_url = trainer.push_to_hub(model_name)
    
    print(f"\n🎉 ¡ÉXITO! Modelo guardado en: {output_path}", flush=True)
    print(f"\n🎉 ¡ÉXITO! Modelo publicado en: {model_url}", flush=True)
        
    utils.clear_gpu_memory()
    return trainer, test_results

In [14]:
trainer_2, test_results_2 = train_model(
    base_model=base_model_2,
    base_tokenizer=base_tokenizer_2,
    train_dataset=train_dataset,
    val_dataset=validation_dataset,
    test_dataset=test_dataset,
    training_args=training_args,
    callbacks=callbacks,
    model_name=trained_model_name_2,
)

🧹 GPU memory cleared
Dataset labels: {0, 1, 2}
Dataset labels: {0, 1, 2}
Dataset labels: {0, 1, 2}


/tmp/ipykernel_88887/1261504697.py:78: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Balanced Accuracy
1,0.819200,0.848892,0.634045,0.608412,0.634045,0.608309,0.479766
2,0.838800,0.759006,0.685734,0.675825,0.685734,0.660436,0.538006
3,0.625200,0.848970,0.692970,0.694702,0.692970,0.675638,0.571212
4,0.786800,0.926982,0.692970,0.698718,0.692970,0.683561,0.596934
5,0.589200,0.885336,0.701930,0.704775,0.701930,0.699777,0.629151
6,0.719000,1.064134,0.704342,0.699442,0.704342,0.701035,0.624881
7,0.680500,1.228325,0.712612,0.713912,0.712612,0.704343,0.618063
8,0.747100,1.323666,0.705720,0.698827,0.705720,0.698253,0.610169
9,0.601100,1.307068,0.708477,0.696461,0.708477,0.694411,0.588486
10,0.591700,1.566560,0.688835,0.692469,0.688835,0.690090,0.624134



🏆 Mejor eval_loss: 0.629151
📍 Alcanzado en época: 11.0


Test Results: {'eval_loss': 1.2231812477111816, 'eval_accuracy': 0.5823293172690763, 'eval_precision': 0.5818432826384515, 'eval_recall': 0.5823293172690763, 'eval_f1': 0.5772865186863301, 'eval_balanced_accuracy': 0.5507327141382868, 'eval_runtime': 3.1356, 'eval_samples_per_second': 317.64, 'eval_steps_per_second': 79.41, 'epoch': 11.0}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 ¡ÉXITO! Modelo guardado en: models/xlm-roberta-sentiment-shp-tokenizer-vocab

🎉 ¡ÉXITO! Modelo publicado en: https://huggingface.co/chechdop/xlm-roberta-sentiment-shp-tokenizer-vocab/tree/main/
🧹 GPU memory cleared
